## 実行準備（最初に実行）
- このNotebookは Colab / Jupyter のどちらでも実行できます。
- まず次のコードセルを実行して、実行環境を確認してください。
- `numpy` と `matplotlib` が未インストールの場合は、コードセルの `%pip` 行を有効化して実行してください。

In [ ]:
# Colab / Jupyter execution prep
import sys

is_colab = "google.colab" in sys.modules
print("Environment:", "Google Colab" if is_colab else "Jupyter / VSCode")
print("If packages are missing, run the next line.")
# %pip install -q numpy>=1.24.0 matplotlib>=3.7.0

# F2 Class Q-Learning: Explanation Notebook

このNotebookは、F2のclass設計（Environment + Agent）をセルごとに確認する教材です。

## 1) GridWorldとは何か
GridWorld は、マス目上でエージェントがゴールを目指すシンプルな環境です。

## 2) state / action / reward の説明
- state: 現在位置
- action: up/down/left/right
- reward: 行動の結果として受け取る値

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# 3) Environment class の役割
class GridWorld:
    def __init__(self, size=(4, 4), goal=(3, 3)):
        self.size = size
        self.goal = goal
        self.state = (0, 0)
        self.actions = ["up", "down", "left", "right"]

    def reset(self):
        self.state = (0, 0)
        return self.state

    def step(self, action):
        x, y = self.state
        if action == "up" and x > 0:
            x -= 1
        elif action == "down" and x < self.size[0] - 1:
            x += 1
        elif action == "left" and y > 0:
            y -= 1
        elif action == "right" and y < self.size[1] - 1:
            y += 1

        self.state = (x, y)
        done = self.state == self.goal
        reward = 1.0 if done else -0.01
        return self.state, reward, done

In [ ]:
# 4) Agent class の役割
class QLearningAgent:
    def __init__(self, env, alpha=0.1, gamma=0.9, epsilon=0.2):
        self.env = env
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.actions = env.actions
        self.q_table = np.zeros((env.size[0], env.size[1], len(env.actions)))
        self.episode_rewards = []

    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.choice(self.actions)
        x, y = state
        return self.actions[np.argmax(self.q_table[x, y])]

    def update(self, state, action, reward, next_state):
        x, y = state
        a = self.actions.index(action)
        nx, ny = next_state
        best_next_q = np.max(self.q_table[nx, ny])
        self.q_table[x, y, a] += self.alpha * (reward + self.gamma * best_next_q - self.q_table[x, y, a])

    def train(self, episodes=500, max_steps=100):
        for _episode in range(episodes):
            state = self.env.reset()
            total_reward = 0.0
            done = False
            steps = 0

            while not done and steps < max_steps:
                action = self.choose_action(state)
                next_state, reward, done = self.env.step(action)
                self.update(state, action, reward, next_state)
                state = next_state
                total_reward += reward
                steps += 1

            self.episode_rewards.append(total_reward)

In [ ]:
env = GridWorld()
agent = QLearningAgent(env)

In [ ]:
# 5) Q-table の形を確認
agent.q_table.shape

In [ ]:
# 6) choose_action() の説明
sample_state = (0, 0)
agent.choose_action(sample_state)

In [ ]:
# 7) update() の説明
state = (0, 0)
action = "right"
next_state, reward, done = env.step(action)
agent.update(state, action, reward, next_state)
agent.q_table[0, 0]

In [ ]:
# 8) train() の実行
agent = QLearningAgent(GridWorld())
agent.train(episodes=600, max_steps=100)
len(agent.episode_rewards)

In [ ]:
# 9) learning curve の可視化
plt.figure(figsize=(7, 4))
plt.plot(agent.episode_rewards, alpha=0.5, label="episode reward")
if len(agent.episode_rewards) >= 30:
    kernel = np.ones(30) / 30
    smooth = np.convolve(agent.episode_rewards, kernel, mode="valid")
    offset = len(agent.episode_rewards) - len(smooth)
    plt.plot(range(offset, len(agent.episode_rewards)), smooth, linewidth=2, label="moving avg (30)")
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("Learning Curve")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 10) Q-table heatmap の可視化
max_q = np.max(agent.q_table, axis=2)
plt.figure(figsize=(5, 4))
plt.imshow(max_q, cmap="YlGnBu", origin="upper")
plt.colorbar(label="max Q")
plt.title("Q-table Heatmap")
plt.xlabel("y")
plt.ylabel("x")
for i in range(max_q.shape[0]):
    for j in range(max_q.shape[1]):
        plt.text(j, i, f"{max_q[i, j]:.2f}", ha="center", va="center")
plt.tight_layout()
plt.show()

In [ ]:
# 11) test_run() による経路確認
def test_run(agent_obj, max_steps=100):
    env_obj = agent_obj.env
    state = env_obj.reset()
    path = [state]
    done = False
    steps = 0

    while not done and steps < max_steps:
        x, y = state
        action = agent_obj.actions[np.argmax(agent_obj.q_table[x, y])]
        state, _reward, done = env_obj.step(action)
        path.append(state)
        steps += 1

    return path

test_path = test_run(agent)
test_path

## 12) TODO
- episodes を変更する
- alpha / gamma / epsilon を変更する
- grid_size を変更する
- reward と goal position を変更する
- learning curve と Q-table heatmap を観察する

## 13) challenge
- 壁やobstacleを追加する
- epsilon decay を実装する
- 最短経路に近づくか確認する
- policy arrow map を作る
- F1関数版とF2クラス版を比較する
- D3倒立振子制御やDeep Q Networkとの関係を調べる